# Reconstruct pool state block-by-block

Turn the irregular swap events into one aligned, gap-free per-pool series
(price + active liquidity), trim to the study window, and save one CSV per pool
under `S.processed_dir`. Parameters come from `arblib.config.STUDY`.

In [1]:
%load_ext autoreload
%autoreload 2

from arblib import data_io, preprocessing as pp
from arblib import formulas as f
from arblib import modeling, regime
from arblib.config import STUDY as S, LIQUIDITY_FILES

REGIME = S.active_regime             # same window as extract (arblib.config)
if S.test_mode:
    win = regime.custom_window(S, S.test_start, S.test_end)
    print(f"[TEST MODE] processing custom window | study starts {win['study_start']} "
          f"(= test_start + {S.test_lead_min} min warm-up)")
else:
    win = regime.study_window(S, REGIME)
    print(f"processing regime: {REGIME} | study starts {win['study_start']}")
STUDY_START = win["study_start"]

processing regime: high | study starts 2026-02-28 00:00:00


## 0. Load the raw swap extracts

In [2]:
dfs = data_io.load_pool_csvs(S.swaps_dir)

Loaded: df_uniswap_swap.csv
               amount0    amount1         dex  evt_block_number  \
0   362865378949565485 -697138212  uniswap_v3          24550518   
1   -54939377945263482  105705703  uniswap_v3          24550519   
2  -219988021276065367  422651373  uniswap_v3          24550519   
3   375257608248799000 -720781973  uniswap_v3          24550519   
4   -52007694331602157  100000000  uniswap_v3          24550520   

                evt_block_time   fee            liquidity  nb_swaps  \
0  2026-02-27 20:00:11.000 UTC   100   480098667751092697         1   
1  2026-02-27 20:00:23.000 UTC  3000  4167707677733689519         1   
2  2026-02-27 20:00:23.000 UTC   100   480098667751092697         3   
3  2026-02-27 20:00:23.000 UTC   500   191041622475999646         1   
4  2026-02-27 20:00:35.000 UTC   100   480098667751092697         4   

                                         pool               sqrtPriceX96  \
0  0xc7bbec68d12a0d1830360f8ec58fa599ba1b0e9b  3472808080342380515

## 1. End-of-block row per pool & block (already done in SQL)

The aggregated swap queries already return one end-of-block row per `(pool, block)` with
`nb_swaps` and `gas_price_max` / `gas_price_med`, so the old `count_swaps` + `clean_all`
(keep-latest-per-block) step is now a pass-through.

In [3]:
# nb_swaps and the single end-of-block row per (pool, block) are already produced by the
# aggregated Dune swap queries, so count_swaps / keep_latest are no longer needed here.
filtered_dfs = dfs

## 2. Split each DEX into one series per pool

In [4]:
pool_dfs = pp.split_by_pool(filtered_dfs)

Created uniswap_1: 39722 rows
Created uniswap_2: 6992 rows
Created uniswap_3: 20410 rows
Created uniswap_4: 303 rows
Created uniswap_5: 1 rows
Created pancake_1: 13970 rows
Created pancake_2: 7687 rows

Total: 7 dataframes
['uniswap_1', 'uniswap_2', 'uniswap_3', 'uniswap_4', 'uniswap_5', 'pancake_1', 'pancake_2']


## 3. Drop pools that trade too rarely to reconstruct

In [5]:
pool_dfs, dropped_pools = pp.filter_pools_by_swap_gap(pool_dfs, S.max_gap_blocks)

k = 6000 blocks
Kept 7 pools

Kept pools:
  uniswap_1: 39722 swaps, max consecutive gap = 6 blocks
  uniswap_2: 6992 swaps, max consecutive gap = 121 blocks
  uniswap_3: 20410 swaps, max consecutive gap = 56 blocks
  uniswap_4: 303 swaps, max consecutive gap = 3228 blocks
  uniswap_5: 1 swaps, max consecutive gap = 0 blocks
  pancake_1: 13970 swaps, max consecutive gap = 56 blocks
  pancake_2: 7687 swaps, max consecutive gap = 67 blocks


## 3b. Drop dynamic-fee pools

The execution-price math needs a single fixed fee per pool, so pools whose `fee`
varies (e.g. Uniswap v4 dynamic-fee hooks) are excluded before saving.

In [6]:
pool_dfs, dropped_fee_pools = pp.filter_pools_by_constant_fee(pool_dfs)

Kept 7 constant-fee pools


## 4. Reconstruct a dense, forward-filled series per pool

In [7]:
reconstructed_pools, global_min, global_max, block_time_map = pp.reconstruct_pool_timeseries(pool_dfs)

Global block range: 24550518 to 24594707
Total blocks: 44190

uniswap_1: 39722 trades -> 44190 blocks (100.0%)
uniswap_2: 6992 trades -> 44189 blocks (100.0%)
uniswap_3: 20410 trades -> 44189 blocks (100.0%)
uniswap_4: 303 trades -> 43421 blocks (98.3%)
uniswap_5: 1 trades -> 2130 blocks (4.8%)
pancake_1: 13970 trades -> 44189 blocks (100.0%)
pancake_2: 7687 trades -> 44189 blocks (100.0%)

Created 7 reconstructed time series


## 4b. Reconstruct active liquidity per block

Rebuild each pool's `liquidity` into the running active-liquidity state using the
mint/burn events (in-range deltas applied between swaps, held constant otherwise).

In [8]:
liq_dfs = data_io.load_pool_csvs(S.liquidity_dir, files=LIQUIDITY_FILES)
reconstructed_pools = pp.reconstruct_liquidity_states(reconstructed_pools, liq_dfs)

Loaded: df_uniswap_liq.csv
          dex  evt_block_number               evt_block_time  evt_index  \
0  uniswap_v3          24550570  2026-02-27 20:10:59.000 UTC         56   
1  uniswap_v3          24550571  2026-02-27 20:11:11.000 UTC         87   
2  uniswap_v3          24550578  2026-02-27 20:12:35.000 UTC        311   
3  uniswap_v3          24550607  2026-02-27 20:18:23.000 UTC        113   
4  uniswap_v3          24550614  2026-02-27 20:19:47.000 UTC        122   

     liquidity_delta                                        pool  tick_lower  \
0     15364635867058  0x4e68ccd3e89f51c3074ca5072bbac773960dfa36     -203100   
1 -18182162782632558  0x4e68ccd3e89f51c3074ca5072bbac773960dfa36     -201360   
2    354480198791459  0x4e68ccd3e89f51c3074ca5072bbac773960dfa36     -202260   
3  12258631573179258  0x4e68ccd3e89f51c3074ca5072bbac773960dfa36     -201060   
4   3092980841578210  0x4e68ccd3e89f51c3074ca5072bbac773960dfa36     -202620   

   tick_upper  
0     -189240  
1     -19

## 4c. Drop pools with invalid (null / zero) active liquidity

A pool with no in-range liquidity (`L = 0`) or undefined `L` (before its first swap) has no executable
price — the round-trip math divides by `L`. Pools with more than `S.max_invalid_liquidity_frac` of
blocks null or `<= 0` are dropped here, before the MEV proxies and save.

In [9]:
reconstructed_pools, dropped_liq_pools = pp.filter_pools_by_liquidity(
    reconstructed_pools, S.max_invalid_liquidity_frac)

Kept 6 pools with <= 10% invalid (null / zero) liquidity
Dropped 1 pool(s):
  uniswap_5: 100.0% of blocks null or <= 0 liquidity


## 5a. MEV friction proxies

Two decay-weighted MEV series per pool: `mev_intensity` (recent top priority tip,
`gas_price_max - base_fee`) and `contest_frequency` (recent rate of same-block races,
`nb_swaps >= 2`), each decayed over past swap blocks with horizon `S.mev_horizon_blocks`.
No forward-fill — a quiet block inherits no stale competition value (gets `NaN`).

In [10]:
chain_gas = data_io.load_chain_gas(S.gas_path)
reconstructed_pools = f.add_mev_intensity(reconstructed_pools, chain_gas, S.mev_horizon_blocks)
reconstructed_pools = f.add_contest_freq(reconstructed_pools, S.mev_horizon_blocks)

uniswap_1: mev_intensity max 2.239e+11 wei
uniswap_2: mev_intensity max 1.500e+11 wei
uniswap_3: mev_intensity max 1.414e+11 wei
uniswap_4: mev_intensity max 7.832e+10 wei
pancake_1: mev_intensity max 1.285e+11 wei
pancake_2: mev_intensity max 3.970e+11 wei
uniswap_1: nb_swaps_ewma max 11.21
uniswap_2: nb_swaps_ewma max 1.48
uniswap_3: nb_swaps_ewma max 4.82
uniswap_4: nb_swaps_ewma max 0.32
pancake_1: nb_swaps_ewma max 2.13
pancake_2: nb_swaps_ewma max 1.50


## 5. Trim to the study window

In [11]:
filtered_pools = pp.filter_by_start_time(reconstructed_pools, STUDY_START)

uniswap_1: 44190 -> 42996 rows
uniswap_2: 44190 -> 42996 rows
uniswap_3: 44190 -> 42996 rows
uniswap_4: 44190 -> 42996 rows
pancake_1: 44190 -> 42996 rows
pancake_2: 44190 -> 42996 rows

Filtered all pools by time >= 2026-02-28 00:00:00


## 6. Save one CSV per pool

In [12]:
data_io.save_processed_pools(filtered_pools, S.processed_dir)

Saved: /Users/matthieu/Downloads/DeFi_limits-to-arbitrage/ethereum/WETH_USDT/high_vol/data_analysis/processed/uniswap_1.csv
Saved: /Users/matthieu/Downloads/DeFi_limits-to-arbitrage/ethereum/WETH_USDT/high_vol/data_analysis/processed/uniswap_2.csv
Saved: /Users/matthieu/Downloads/DeFi_limits-to-arbitrage/ethereum/WETH_USDT/high_vol/data_analysis/processed/uniswap_3.csv
Saved: /Users/matthieu/Downloads/DeFi_limits-to-arbitrage/ethereum/WETH_USDT/high_vol/data_analysis/processed/uniswap_4.csv
Saved: /Users/matthieu/Downloads/DeFi_limits-to-arbitrage/ethereum/WETH_USDT/high_vol/data_analysis/processed/pancake_1.csv
Saved: /Users/matthieu/Downloads/DeFi_limits-to-arbitrage/ethereum/WETH_USDT/high_vol/data_analysis/processed/pancake_2.csv
Done.


## 7. Save the common (pool-independent) modeling covariates

Persist the covariates every pool pair shares to `S.common_covariates_dir`:

- **`CEX_volatility.parquet`** — `[time, ewma_vol]`: RiskMetrics EWMA volatility of the `token0/token1`
  rate on a minute grid (`λ = exp(-1/S.vol_horizon_min)`), joined to blocks by a backward merge on
  `time` downstream.
- **`chain_covariates.parquet`** — `[block_number, time, log_base_fee_per_gas, gas_util,
  log1p_tip_p50, log1p_tip_p90]`, joined by exact `block_number`.

Both span the full extract window; downstream joins select the study blocks.

In [13]:
x_usd = data_io.load_price_series(S.x_price_path)
y_usd = data_io.load_price_series(S.y_price_path)

modeling.save_common_covariates(x_usd, y_usd, chain_gas, S.common_covariates_dir, S.vol_horizon_min)
S.pair_covariates_dir.mkdir(parents=True, exist_ok=True)

Saved common covariates (CEX_volatility, chain_covariates) to /Users/matthieu/Downloads/DeFi_limits-to-arbitrage/ethereum/WETH_USDT/high_vol/modeling/covariates/common_covariates
